# Final paper results reporting

This notebook does not compute new scientific results. It only transforms frozen experimental and statistical artifacts into publication-ready tables and figures.

Official statistical run: fc7bbae7bd5d0c99

The notebook is read-only with respect to scientific artifacts. It does not train, translate, infer, bootstrap, recompute thresholds, load models or checkpoints, initialize CUDA, or access the internet. It writes only deterministic reporting outputs under results/paper/.

## 1. Provenance and frozen inputs

All scientific values are resolved from the source registry below. Constants are limited to presentation labels, condition order, expected provenance checks, and output paths.

In [1]:
from pathlib import Path
import json
from datetime import datetime
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "notebooks").is_dir(), f"Could not resolve repository root from {ROOT}"

OFFICIAL_RUN_ID = "fc7bbae7bd5d0c99"
SUPERSEDED_RUN_ID = "01a3d72bd8e55038"

REGISTRY = {
    "OFFICIAL_BOOTSTRAP_RUN": ROOT / "runs/publichearing_final_paired_bootstrap" / OFFICIAL_RUN_ID,
    "MAIN_SUMMARY": ROOT / "runs/publichearing_final_paired_bootstrap" / OFFICIAL_RUN_ID / "final_bootstrap_summary.csv",
    "FINAL_REPORT": ROOT / "runs/publichearing_final_paired_bootstrap" / OFFICIAL_RUN_ID / "final_bootstrap_report.md",
    "MANIFEST": ROOT / "runs/publichearing_final_paired_bootstrap" / OFFICIAL_RUN_ID / "manifest.json",
    "RESOLVED_CONFIG": ROOT / "runs/publichearing_final_paired_bootstrap" / OFFICIAL_RUN_ID / "resolved_config.json",
    "POPULATION_VALIDATION": ROOT / "runs/publichearing_final_paired_bootstrap" / OFFICIAL_RUN_ID / "population_validation.json",
    "POOLING_RUN": ROOT / "runs/publichearing_pooling_paired_bootstrap/b92d4fd6a6a7e771",
    "BASELINE_RUN": ROOT / "runs/publichearing_off_the_shelf_max_entailment/54d9c623f8685c39",
    "SUPERVISED_METRICS": ROOT / "results/publichearing_lora_attention_mil/741ed3152c2175e7/outputs/overall_oof_metrics.csv",
    "FILTERED_AGGREGATE": ROOT / "runs/ragtruth_pt_nllb_filtered_confirmatory/63745412afdb52ac/aggregate/aggregate_metrics.csv",
    "BERTIMBAU_AGGREGATE": ROOT / "runs/ragtruth_pt_nllb_bertimbau_confirmatory/fce6272e2e72726d/aggregate/aggregate_metrics.csv",
    "PT_NLLB_THRESHOLDED": ROOT / "runs/publichearing_pt_nllb_attention_vs_set_thresholded_bootstrap/e0c75270065fc471/bootstrap_summary.json",
}
SUPERSEDED_BOOTSTRAP_RUN = ROOT / "runs/publichearing_final_paired_bootstrap" / SUPERSEDED_RUN_ID

OUTPUT_ROOT = ROOT / "results/paper"
TABLE_ROOT = OUTPUT_ROOT / "tables"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
APPENDIX_ROOT = OUTPUT_ROOT / "appendix"
for directory in (OUTPUT_ROOT, TABLE_ROOT, FIGURE_ROOT, APPENDIX_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(Path(path).read_text())

def require_files(*keys):
    missing = [f"{key}: {REGISTRY[key]}" for key in keys if not REGISTRY[key].exists()]
    assert not missing, "Required frozen input(s) missing:\n" + "\n".join(missing)

require_files(
    "OFFICIAL_BOOTSTRAP_RUN", "MAIN_SUMMARY", "FINAL_REPORT", "MANIFEST",
    "RESOLVED_CONFIG", "POPULATION_VALIDATION", "POOLING_RUN", "BASELINE_RUN",
    "SUPERVISED_METRICS", "FILTERED_AGGREGATE", "BERTIMBAU_AGGREGATE", "PT_NLLB_THRESHOLDED",
)
assert REGISTRY["OFFICIAL_BOOTSTRAP_RUN"].name == OFFICIAL_RUN_ID
assert REGISTRY["OFFICIAL_BOOTSTRAP_RUN"] != SUPERSEDED_BOOTSTRAP_RUN
assert str(SUPERSEDED_BOOTSTRAP_RUN) not in {
    str(path) for path in REGISTRY.values()
}, "The superseded run must never be registered as a source."

MANIFEST = read_json(REGISTRY["MANIFEST"])
CONFIG = read_json(REGISTRY["RESOLVED_CONFIG"])
POPULATION = read_json(REGISTRY["POPULATION_VALIDATION"])
SUMMARY = pd.read_csv(REGISTRY["MAIN_SUMMARY"])
assert MANIFEST["signature"] == OFFICIAL_RUN_ID
assert MANIFEST["status"] == "completed"
print("Registry loaded; scientific inputs remain frozen and read-only.")

Registry loaded; scientific inputs remain frozen and read-only.


## 2. Load and validate final artifacts

The final summary is the primary source for point estimates and paired confidence intervals. Other sources are used only for validated aggregates that are not present in the final summary.

In [2]:
EXPECTED = {
    "examples": 4235, "positives": 501, "hearings": 206,
    "replicates": 10000, "seed": 20260815,
    "ci_method": "percentile", "confidence_level": 0.95,
}
assert set(SUMMARY["replicates"].unique()) == {EXPECTED["replicates"]}
assert set(SUMMARY["valid_replicates"].unique()) == {EXPECTED["replicates"]}
assert set(SUMMARY["invalid_replicates"].unique()) == {0}
assert set(SUMMARY["seed"].unique()) == {EXPECTED["seed"]}
assert set(SUMMARY["population_examples"].unique()) == {EXPECTED["examples"]}
assert set(SUMMARY["population_positives"].unique()) == {EXPECTED["positives"]}
assert set(SUMMARY["population_hearings"].unique()) == {EXPECTED["hearings"]}
assert POPULATION["examples"] == EXPECTED["examples"]
assert POPULATION["positives"] == EXPECTED["positives"]
assert POPULATION["hearings"] == EXPECTED["hearings"]
assert POPULATION["exact_example_alignment"]
assert POPULATION["exact_label_alignment"]
assert POPULATION["exact_hearing_alignment"]
assert POPULATION["probabilities_valid"]
assert CONFIG["bootstrap"]["n_replicates"] == EXPECTED["replicates"]
assert CONFIG["bootstrap"]["seed"] == EXPECTED["seed"]
assert CONFIG["bootstrap"]["ci_method"] == EXPECTED["ci_method"]
assert CONFIG["bootstrap"]["confidence_level"] == EXPECTED["confidence_level"]

HIGHER_BETTER = {"auprc", "auroc", "f1", "mcc", "precision", "recall", "balanced_accuracy"}
LOWER_BETTER = {"brier", "fpr"}

def expected_significance(row):
    low, high = float(row["ci_low"]), float(row["ci_high"])
    family, metric = row["contrast_family"], row["metric"]
    if family in {"translation_interaction", "set_effect_interaction"}:
        return "NEGATIVE" if high < 0 else "POSITIVE" if low > 0 else "INCONCLUSIVE"
    if metric in HIGHER_BETTER:
        return "FAVOR_B" if low > 0 else "FAVOR_A" if high < 0 else "INCONCLUSIVE"
    if metric in LOWER_BETTER:
        return "FAVOR_A" if low > 0 else "FAVOR_B" if high < 0 else "INCONCLUSIVE"
    raise AssertionError(f"Unknown metric orientation: {metric}")

orientation_mismatches = []
for idx, row in SUMMARY.iterrows():
    expected = expected_significance(row)
    if row["significant_direction"] != expected:
        orientation_mismatches.append({"row": int(idx), "expected": expected, "actual": row["significant_direction"]})
assert not orientation_mismatches, f"Inconsistent significant_direction values: {orientation_mismatches}"

CONDITIONS = pd.DataFrame([
    {"condition_id": "en_ph_pt", "training": "EN", "target": "PH PT", "attention_id": "en_attention_ph_pt", "set_id": "en_set_ph_pt", "label": "EN → PH PT"},
    {"condition_id": "pt_nllb_ph_pt", "training": "PT-NLLB", "target": "PH PT", "attention_id": "pt_nllb_attention_ph_pt", "set_id": "pt_nllb_set_ph_pt", "label": "PT-NLLB → PH PT"},
    {"condition_id": "pt_madlad_ph_pt", "training": "PT-MADLAD", "target": "PH PT", "attention_id": "pt_madlad_attention_ph_pt", "set_id": "pt_madlad_set_ph_pt", "label": "PT-MADLAD → PH PT"},
    {"condition_id": "en_ph_en_nllb", "training": "EN", "target": "PH EN-NLLB", "attention_id": "en_attention_ph_en_nllb", "set_id": "en_set_ph_en_nllb", "label": "EN → PH EN-NLLB"},
    {"condition_id": "en_ph_en_madlad", "training": "EN", "target": "PH EN-MADLAD", "attention_id": "en_attention_ph_en_madlad", "set_id": "en_set_ph_en_madlad", "label": "EN → PH EN-MADLAD"},
])
AGGREGATOR_CONTRASTS = {
    "en_ph_pt": "en_set_vs_attention_ph_pt",
    "pt_nllb_ph_pt": "pt_nllb_set_vs_attention_ph_pt",
    "pt_madlad_ph_pt": "pt_madlad_set_vs_attention_ph_pt",
    "en_ph_en_nllb": "en_set_vs_attention_ph_en_nllb",
    "en_ph_en_madlad": "en_set_vs_attention_ph_en_madlad",
}

def summary_rows(family=None, metric=None, regime=None):
    frame = SUMMARY
    if family is not None: frame = frame.loc[frame["contrast_family"].eq(family)]
    if metric is not None: frame = frame.loc[frame["metric"].eq(metric)]
    if regime is not None: frame = frame.loc[frame["regime"].eq(regime)]
    return frame.copy()

POINTS = {}
for _, row in SUMMARY.loc[SUMMARY["regime"].eq("threshold_free")].dropna(subset=["point_a", "point_b"]).iterrows():
    for condition_id, value in ((row["condition_a"], row["point_a"]), (row["condition_b"], row["point_b"])):
        POINTS.setdefault(condition_id, {}).setdefault(row["metric"], set()).add(float(value))
for condition_id, metrics in POINTS.items():
    for metric, values in metrics.items():
        assert len(values) == 1, f"Conflicting frozen point estimates for {condition_id}/{metric}: {values}"
POINTS = {cid: {metric: next(iter(values)) for metric, values in metrics.items()} for cid, metrics in POINTS.items()}

def point(condition_id, metric):
    assert condition_id in POINTS and metric in POINTS[condition_id], f"Missing frozen point: {condition_id}/{metric}"
    return POINTS[condition_id][metric]

def contrast_row(contrast_id, metric, regime="threshold_free", family=None):
    rows = SUMMARY.loc[
        SUMMARY["contrast_id"].eq(contrast_id) & SUMMARY["metric"].eq(metric) & SUMMARY["regime"].eq(regime)
    ]
    if family is not None: rows = rows.loc[rows["contrast_family"].eq(family)]
    assert len(rows) == 1, f"Expected one row for {contrast_id}/{metric}/{regime}, found {len(rows)}"
    return rows.iloc[0]

def fmt(value):
    return f"{float(value):.4f}"

def fmt_signed(value):
    return f"{float(value):+.4f}"

def ci_text(row):
    return f"[{fmt(row['ci_low'])}, {fmt(row['ci_high'])}]"

def verdict(row):
    return {"FAVOR_B": "Favors Set", "FAVOR_A": "Favors Attention", "INCONCLUSIVE": "Inconclusive"}.get(row["significant_direction"], row["significant_direction"])

def save_table(frame, csv_path, tex_path, caption=None, column_format=None):
    frame.to_csv(csv_path, index=False)
    tex = frame.to_latex(index=False, escape=False, caption=caption, column_format=column_format)
    Path(tex_path).write_text(tex)
    assert Path(csv_path).stat().st_size > 0 and Path(tex_path).stat().st_size > 0

display(pd.DataFrame([{
    "official bootstrap run": OFFICIAL_RUN_ID, "final_bootstrap_summary.csv": str(REGISTRY["MAIN_SUMMARY"]),
    "final_bootstrap_report.md": str(REGISTRY["FINAL_REPORT"]), "manifest": str(REGISTRY["MANIFEST"]),
    "bootstrap seed": EXPECTED["seed"], "replicates": EXPECTED["replicates"], "CI": "percentile 95%",
    "examples": EXPECTED["examples"], "positives": EXPECTED["positives"], "hearings": EXPECTED["hearings"],
    "orientation mismatches": len(orientation_mismatches),
}]))
print("Provenance and frozen-input assertions passed.")

,official bootstrap run,final_bootstrap_summary.csv,final_bootstrap_report.md,manifest,bootstrap seed,replicates,CI,examples,positives,hearings,orientation mismatches
0,fc7bbae7bd5d0c99,/home/guto/Development/python/ragtruth_publich...,/home/guto/Development/python/ragtruth_publich...,/home/guto/Development/python/ragtruth_publich...,20260815,10000,percentile 95%,4235,501,206,0


Provenance and frozen-input assertions passed.


## 3. Main transfer results

Table 1 — Cross-lingual transfer and translation strategies on PublicHearingBR.

In [3]:
table1 = pd.DataFrame([
    {"Training": row.training, "Target": row.target,
     "Attention AUPRC": point(row.attention_id, "auprc"), "Set AUPRC": point(row.set_id, "auprc"),
     "Attention AUROC": point(row.attention_id, "auroc"), "Set AUROC": point(row.set_id, "auroc")}
    for row in CONDITIONS.itertuples()
])
assert len(table1) == 5
assert (table1["Set AUPRC"] >= table1["Attention AUPRC"]).all()

table1_csv = table1.copy()
for col in table1_csv.columns[2:]: table1_csv[col] = table1_csv[col].map(fmt)
table1_tex = table1_csv.copy()
for idx in table1.index:
    for acol, bcol in [("Attention AUPRC", "Set AUPRC"), ("Attention AUROC", "Set AUROC")]:
        a, b = table1.loc[idx, acol], table1.loc[idx, bcol]
        table1_tex.loc[idx, acol] = rf"\textbf{{{fmt(a)}}}" if a > b else fmt(a)
        table1_tex.loc[idx, bcol] = rf"\textbf{{{fmt(b)}}}" if b >= a else fmt(b)
table1_csv.to_csv(TABLE_ROOT / "table_main_transfer.csv", index=False)
(TABLE_ROOT / "table_main_transfer.tex").write_text(table1_tex.to_latex(index=False, escape=False, caption="Cross-lingual transfer and translation strategies on PublicHearingBR", column_format="llrrrr"))
assert (TABLE_ROOT / "table_main_transfer.csv").stat().st_size > 0 and (TABLE_ROOT / "table_main_transfer.tex").stat().st_size > 0
display(table1_csv)

fig, ax = plt.subplots(figsize=(7.0, 4.6))
y = np.arange(len(CONDITIONS))[::-1]
a = table1["Attention AUPRC"].to_numpy()
b = table1["Set AUPRC"].to_numpy()
ax.hlines(y, a, b, color="0.75", linewidth=1.2, zorder=1)
ax.scatter(a, y, marker="o", s=42, color="#4C78A8", label="Attention", zorder=3)
ax.scatter(b, y, marker="D", s=38, color="#F58518", label="Set Transformer", zorder=3)
ax.set_yticks(y, CONDITIONS["label"].tolist())
ax.set_xlabel("AUPRC")
ax.set_xlim(max(0, table1[["Attention AUPRC", "Set AUPRC"]].min().min() - 0.04), table1[["Attention AUPRC", "Set AUPRC"]].max().max() + 0.04)
ax.grid(axis="x", color="0.90", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "figure_transfer_overview.pdf", bbox_inches="tight", metadata={"Creator": "Final paper results reporting", "CreationDate": datetime(2020, 1, 1)})
fig.savefig(FIGURE_ROOT / "figure_transfer_overview.png", dpi=300, bbox_inches="tight")
plt.close(fig)

table2_rows = []
for row in CONDITIONS.itertuples():
    result = contrast_row(AGGREGATOR_CONTRASTS[row.condition_id], "auprc", family="aggregator")
    table2_rows.append({"Condition": row.label, "Attention AUPRC": float(result["point_a"]), "Set AUPRC": float(result["point_b"]),
                        "Delta Set-Attention": float(result["observed_delta"]), "95% CI": ci_text(result),
                        "Verdict": verdict(result), "_ci_low": float(result["ci_low"]), "_ci_high": float(result["ci_high"])})
table2 = pd.DataFrame(table2_rows)
assert len(table2) == 5 and (table2["Delta Set-Attention"] > 0).all()
assert int((table2["_ci_low"] > 0).sum()) == 4
assert int(((table2["_ci_low"] <= 0) & (table2["_ci_high"] >= 0)).sum()) == 1
table2_export = table2.drop(columns=["_ci_low", "_ci_high"]).copy()
for col in ["Attention AUPRC", "Set AUPRC", "Delta Set-Attention"]:
    table2_export[col] = table2_export[col].map(fmt_signed if col == "Delta Set-Attention" else fmt)
save_table(table2_export, TABLE_ROOT / "table_set_vs_attention.csv", TABLE_ROOT / "table_set_vs_attention.tex", "Paired grouped bootstrap for Set Transformer vs gated attention", "lrrrrl")
display(table2_export)

translation_specs = [
    ("Training-side translation", "PT-NLLB training − EN training", "pt_nllb_set_vs_en_ph_pt", "training_translation"),
    ("Training-side translation", "PT-MADLAD training − EN training", "pt_madlad_set_vs_en_ph_pt", "training_translation"),
    ("Target-side translation", "PH EN-NLLB target − PH PT target", "en_set_ph_en_nllb_vs_pt", "target_translation"),
    ("Target-side translation", "PH EN-MADLAD target − PH PT target", "en_set_ph_en_madlad_vs_pt", "target_translation"),
]
translation_effects = []
for group, label, cid, family in translation_specs:
    r = contrast_row(cid, "auprc", family=family)
    translation_effects.append({"Group": group, "Contrast": label, "Observed delta": float(r["observed_delta"]), "CI low": float(r["ci_low"]), "CI high": float(r["ci_high"])})
translation_effects = pd.DataFrame(translation_effects)
fig, ax = plt.subplots(figsize=(7.2, 4.5))
yp = np.array([3, 2, 1, 0])
ax.errorbar(translation_effects["Observed delta"], yp,
            xerr=[translation_effects["Observed delta"] - translation_effects["CI low"], translation_effects["CI high"] - translation_effects["Observed delta"]],
            fmt="o", color="#4C78A8", ecolor="#4C78A8", elinewidth=1.5, capsize=3, markersize=5)
ax.axvline(0, color="0.25", linewidth=1.0)
ax.axhline(1.5, color="0.85", linewidth=1.0)
ax.set_yticks(yp, translation_effects["Contrast"].tolist())
ax.set_xlabel("AUPRC difference (Set Transformer)")
ax.text(0.01, 0.98, "Training-side translation", transform=ax.transAxes, va="top", fontsize=9, color="0.35")
ax.text(0.01, 0.47, "Target-side translation", transform=ax.transAxes, va="top", fontsize=9, color="0.35")
ax.grid(axis="x", color="0.90", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "figure_translation_effects.pdf", bbox_inches="tight", metadata={"Creator": "Final paper results reporting", "CreationDate": datetime(2020, 1, 1)})
fig.savefig(FIGURE_ROOT / "figure_translation_effects.png", dpi=300, bbox_inches="tight")
plt.close(fig)

interaction_specs = [("NLLB", "nllb_set_target_minus_training"), ("MADLAD", "madlad_set_target_minus_training")]
table3_rows, table3_full_rows = [], []
for translator, cid in interaction_specs:
    for metric in ["auprc", "auroc", "brier"]:
        r = contrast_row(cid, metric, regime="interaction", family="translation_interaction")
        item = {"Translator": translator, "Metric": metric.upper(), "Interaction": float(r["observed_delta"]),
                "95% CI": ci_text(r), "Interpretation": "Target-side more harmful" if r["significant_direction"] == "NEGATIVE" else "Inconclusive"}
        table3_full_rows.append(item)
        if metric in {"auprc", "auroc"}: table3_rows.append(item)
table3 = pd.DataFrame(table3_rows)
table3_full = pd.DataFrame(table3_full_rows)
for frame, stem in [(table3, "table_translation_interaction"), (table3_full, "table_translation_interaction_full")]:
    out = frame.copy()
    out["Interaction"] = out["Interaction"].map(fmt_signed)
    save_table(out, TABLE_ROOT / f"{stem}.csv", TABLE_ROOT / f"{stem}.tex", "Paired interaction between training-side and target-side translation effects", "llrrl")
display(table3.assign(Interaction=table3["Interaction"].map(fmt_signed)))

,Training,Target,Attention AUPRC,Set AUPRC,Attention AUROC,Set AUROC
0,EN,PH PT,0.5949,0.6035,0.8788,0.8862
1,PT-NLLB,PH PT,0.5370,0.5658,0.8664,0.8749
2,PT-MADLAD,PH PT,0.5593,0.5887,0.8693,0.8749
3,EN,PH EN-NLLB,0.3900,0.4062,0.8277,0.8351
4,EN,PH EN-MADLAD,0.2179,0.2332,0.7216,0.7297


,Condition,Attention AUPRC,Set AUPRC,Delta Set-Attention,95% CI,Verdict
0,EN → PH PT,0.5949,0.6035,+0.0086,"[-0.0046, 0.0231]",Inconclusive
1,PT-NLLB → PH PT,0.5370,0.5658,+0.0289,"[0.0130, 0.0442]",Favors Set
2,PT-MADLAD → PH PT,0.5593,0.5887,+0.0294,"[0.0149, 0.0433]",Favors Set
3,EN → PH EN-NLLB,0.3900,0.4062,+0.0162,"[0.0003, 0.0335]",Favors Set
4,EN → PH EN-MADLAD,0.2179,0.2332,+0.0153,"[0.0056, 0.0252]",Favors Set


,Translator,Metric,Interaction,95% CI,Interpretation
0,NLLB,AUPRC,-0.1596,"[-0.1914, -0.1224]",Target-side more harmful
1,NLLB,AUROC,-0.0398,"[-0.0524, -0.0273]",Target-side more harmful
2,MADLAD,AUPRC,-0.3555,"[-0.3952, -0.3082]",Target-side more harmful
3,MADLAD,AUROC,-0.1452,"[-0.1674, -0.1228]",Target-side more harmful


## 4. Set Transformer effect

The Set-versus-Attention table and transfer overview above use paired frozen summary rows. Bold in Table 1 marks the larger value within each metric pair only; it does not encode significance.

## 5. Translation-side effects

Figure 2 separates training-side and target-side contrasts on one forest plot. The zero line is a reference for the observed delta.

## 6. Paired interactions

Table 3 uses the validated definition in which a negative AUPRC/AUROC interaction means target-side translation is associated with larger degradation than training-side translation.

## 7. Contextual baselines

The supervised target-domain result is descriptive context only. No paired comparison, p-value, equivalence, non-inferiority, or significance annotation is produced for EN + Set versus supervised Attention.

In [4]:
baseline_metrics = read_json(REGISTRY["BASELINE_RUN"] / "metrics.json")
supervised = pd.read_csv(REGISTRY["SUPERVISED_METRICS"])
supervised_ranking = supervised.loc[supervised["criterion"].eq("ranking")].iloc[0]
table4 = pd.DataFrame([
    {"Method": "Off-the-shelf NLI", "Target supervision": "No", "AUPRC": float(baseline_metrics["AUPRC"]), "AUROC": float(baseline_metrics["AUROC"])},
    {"Method": "EN + Attention", "Target supervision": "No", "AUPRC": point("en_attention_ph_pt", "auprc"), "AUROC": point("en_attention_ph_pt", "auroc")},
    {"Method": "EN + Set", "Target supervision": "No", "AUPRC": point("en_set_ph_pt", "auprc"), "AUROC": point("en_set_ph_pt", "auroc")},
    {"Method": "Target-supervised Attention", "Target supervision": "Yes (out-of-fold)", "AUPRC": float(supervised_ranking["AUPRC"]), "AUROC": float(supervised_ranking["AUROC"])},
])
table4_export = table4.copy()
for col in ["AUPRC", "AUROC"]: table4_export[col] = table4_export[col].map(fmt)
save_table(table4_export, TABLE_ROOT / "table_baselines.csv", TABLE_ROOT / "table_baselines.tex", "Contextual baselines on PublicHearingBR", "llrr")
with (TABLE_ROOT / "table_baselines.tex").open("a") as handle:
    handle.write("\\par\\smallskip\\noindent\\textit{The supervised model follows an out-of-fold target-domain protocol and is included only as descriptive context, not as a paired statistical comparison.}\\n")
display(table4_export)
print("supervised comparison descriptive only: YES")

,Method,Target supervision,AUPRC,AUROC
0,Off-the-shelf NLI,No,0.3375,0.7680
1,EN + Attention,No,0.5949,0.8788
2,EN + Set,No,0.6035,0.8862
3,Target-supervised Attention,Yes (out-of-fold),0.6044,0.8851


supervised comparison descriptive only: YES


## 8. Appendix tables

The appendix exports include pooling, negative ablations, calibration, thresholded operating metrics, the full threshold-free paired summary, and Set-effect interactions.

In [5]:
def aggregate_metrics(path):
    frame = pd.read_csv(path)
    rows = frame.loc[frame["dataset"].eq("publichearing_zero_shot") & frame["threshold_regime"].eq("threshold_free")]
    assert len(rows) == 1, f"Expected one threshold-free zero-shot row in {path}, found {len(rows)}"
    row = rows.iloc[0]
    return {metric: float(row[metric]) for metric in ["AUPRC", "AUROC", "Brier"]}

pooling_observed = read_json(REGISTRY["POOLING_RUN"] / "observed_metrics.json")["campaigns"]
pooling = pd.DataFrame([
    {"Pooling": "Mean", "AUPRC": pooling_observed["mean"]["mean_across_seeds"]["auprc"], "AUROC": pooling_observed["mean"]["mean_across_seeds"]["auroc"], "Brier": pooling_observed["mean"]["mean_across_seeds"]["brier"]},
    {"Pooling": "Max", "AUPRC": pooling_observed["max"]["mean_across_seeds"]["auprc"], "AUROC": pooling_observed["max"]["mean_across_seeds"]["auroc"], "Brier": pooling_observed["max"]["mean_across_seeds"]["brier"]},
    {"Pooling": "Attention", "AUPRC": point("en_attention_ph_pt", "auprc"), "AUROC": point("en_attention_ph_pt", "auroc"), "Brier": point("en_attention_ph_pt", "brier")},
    {"Pooling": "Set", "AUPRC": point("en_set_ph_pt", "auprc"), "AUROC": point("en_set_ph_pt", "auroc"), "Brier": point("en_set_ph_pt", "brier")},
])
pooling_export = pooling.copy()
for col in pooling_export.columns[1:]: pooling_export[col] = pooling_export[col].map(fmt)
save_table(pooling_export, APPENDIX_ROOT / "table_pooling_ablations.csv", APPENDIX_ROOT / "table_pooling_ablations.tex", "Pooling ablations for EN to PH PT", "lrrr")

filtered = aggregate_metrics(REGISTRY["FILTERED_AGGREGATE"])
bertimbau = aggregate_metrics(REGISTRY["BERTIMBAU_AGGREGATE"])
negative = pd.DataFrame([
    {"Method": "Off-the-shelf NLI", "AUPRC": baseline_metrics["AUPRC"], "AUROC": baseline_metrics["AUROC"], "Brier": baseline_metrics["Brier"]},
    {"Method": "EN + Attention", "AUPRC": point("en_attention_ph_pt", "auprc"), "AUROC": point("en_attention_ph_pt", "auroc"), "Brier": point("en_attention_ph_pt", "brier")},
    {"Method": "PT-NLLB filtered + Attention", "AUPRC": filtered["AUPRC"], "AUROC": filtered["AUROC"], "Brier": filtered["Brier"]},
    {"Method": "PT-NLLB + BERTimbau + Attention", "AUPRC": bertimbau["AUPRC"], "AUROC": bertimbau["AUROC"], "Brier": bertimbau["Brier"]},
])
negative_export = negative.copy()
for col in negative_export.columns[1:]: negative_export[col] = negative_export[col].map(fmt)
save_table(negative_export, APPENDIX_ROOT / "table_negative_ablations.csv", APPENDIX_ROOT / "table_negative_ablations.tex", "Negative ablations and contextual references", "lrrr")

brier = pd.DataFrame([{"Condition": row.label, "Attention Brier": point(row.attention_id, "brier"), "Set Brier": point(row.set_id, "brier")} for row in CONDITIONS.itertuples()])
brier_export = brier.copy()
for col in brier_export.columns[1:]: brier_export[col] = brier_export[col].map(fmt)
save_table(brier_export, APPENDIX_ROOT / "table_brier_main_conditions.csv", APPENDIX_ROOT / "table_brier_main_conditions.tex", "Brier scores for the five main conditions", "lrr")

thresholded = SUMMARY.loc[SUMMARY["contrast_family"].eq("thresholded_set_attention") & SUMMARY["regime"].isin(["best_f1", "fpr10"])].copy()
legacy = read_json(REGISTRY["PT_NLLB_THRESHOLDED"])
legacy_rows = []
for regime in ["best_f1", "fpr10"]:
    for metric, values in legacy[regime]["metrics"].items():
        if metric not in {"precision", "recall", "f1", "mcc", "fpr"}: continue
        b = values["bootstrap"]
        if metric == "fpr":
            direction = "FAVOR_A" if b["ci_lower"] > 0 else "FAVOR_B" if b["ci_upper"] < 0 else "INCONCLUSIVE"
        else:
            direction = "FAVOR_B" if b["ci_lower"] > 0 else "FAVOR_A" if b["ci_upper"] < 0 else "INCONCLUSIVE"
        legacy_rows.append({
            "contrast_id": "pt_nllb_set_vs_attention_ph_pt_thresholded", "contrast_family": "thresholded_set_attention",
            "regime": regime, "metric": metric, "point_a": values["attention_mean"], "point_b": values["set_mean"],
            "observed_delta": values["observed_delta"], "ci_low": b["ci_lower"], "ci_high": b["ci_upper"],
            "probability_favorable": b["favorable_probability"], "significant_direction": direction,
        })
thresholded = pd.concat([thresholded, pd.DataFrame(legacy_rows)], ignore_index=True)
thresholded["condition_id"] = thresholded["contrast_id"].str.replace("_thresholded", "", regex=False).map({v: k for k, v in AGGREGATOR_CONTRASTS.items()})
thresholded["order"] = thresholded["condition_id"].map({cid: i for i, cid in enumerate(CONDITIONS["condition_id"])})
assert len(thresholded) == 5 * 2 * 5
thresholded["Condition"] = thresholded["condition_id"].map(dict(zip(CONDITIONS["condition_id"], CONDITIONS["label"])))
thresholded_export = thresholded.sort_values(["order", "regime", "metric"])[["Condition", "regime", "metric", "point_a", "point_b", "observed_delta", "ci_low", "ci_high", "probability_favorable", "significant_direction"]].rename(columns={"regime": "Regime", "metric": "Metric", "point_a": "Attention", "point_b": "Set", "observed_delta": "Delta Set-Attention", "ci_low": "CI low", "ci_high": "CI high", "probability_favorable": "Probability favorable", "significant_direction": "Verdict"})
thresholded_export["Verdict"] = thresholded_export["Verdict"].map({"FAVOR_A": "Favors Attention", "FAVOR_B": "Favors Set", "INCONCLUSIVE": "Inconclusive"})
thresholded_display = thresholded_export.copy()
for col in ["Attention", "Set", "Delta Set-Attention", "CI low", "CI high", "Probability favorable"]:
    thresholded_display[col] = thresholded_display[col].map(fmt_signed if col == "Delta Set-Attention" else fmt)
save_table(thresholded_display, APPENDIX_ROOT / "table_thresholded_results.csv", APPENDIX_ROOT / "table_thresholded_results.tex", "Thresholded paired results for the five main conditions", "llrrrrrrrl")
with (APPENDIX_ROOT / "table_thresholded_results.tex").open("a") as handle:
    handle.write("\\par\\smallskip\\noindent\\textit{FPR is lower-is-better; the verdict uses the corrected significant direction from the frozen artifact.}\\n")

full = SUMMARY.loc[SUMMARY["regime"].eq("threshold_free")].copy()
full_export = full[["contrast_family", "condition_a", "condition_b", "metric", "point_a", "point_b", "observed_delta", "ci_low", "ci_high", "probability_favorable", "significant_direction"]].rename(columns={"contrast_family": "Contrast family", "condition_a": "Condition A", "condition_b": "Condition B", "metric": "Metric", "point_a": "Point A", "point_b": "Point B", "observed_delta": "Delta", "ci_low": "CI low", "ci_high": "CI high", "probability_favorable": "Probability favorable", "significant_direction": "Verdict"})
full_export["Verdict"] = full_export["Verdict"].map({"FAVOR_A": "Favors A", "FAVOR_B": "Favors B", "INCONCLUSIVE": "Inconclusive", "NEGATIVE": "Negative", "POSITIVE": "Positive"})
full_display = full_export.copy()
for col in ["Point A", "Point B", "Delta", "CI low", "CI high", "Probability favorable"]:
    full_display[col] = full_display[col].map(lambda x: "" if pd.isna(x) else fmt_signed(x) if col == "Delta" else fmt(x))
save_table(full_display, APPENDIX_ROOT / "table_full_paired_bootstrap.csv", APPENDIX_ROOT / "table_full_paired_bootstrap.tex", "Full threshold-free paired bootstrap summary", "llllrrrrrl")

set_effect = SUMMARY.loc[SUMMARY["contrast_family"].eq("set_effect_interaction") & SUMMARY["metric"].eq("auprc")].copy()
set_effect["Translator"] = set_effect["contrast_id"].str.extract(r"^(nllb|madlad)")[0].str.upper()
set_effect_export = set_effect[["Translator", "observed_delta", "ci_low", "ci_high", "significant_direction"]].rename(columns={"observed_delta": "AUPRC interaction", "ci_low": "CI low", "ci_high": "CI high", "significant_direction": "Verdict"})
set_effect_export["Verdict"] = set_effect_export["Verdict"].map({"INCONCLUSIVE": "Inconclusive", "NEGATIVE": "Negative", "POSITIVE": "Positive"})
set_effect_display = set_effect_export.copy()
for col in ["AUPRC interaction", "CI low", "CI high"]: set_effect_display[col] = set_effect_display[col].map(fmt_signed)
save_table(set_effect_display, APPENDIX_ROOT / "table_set_effect_interactions.csv", APPENDIX_ROOT / "table_set_effect_interactions.tex", "Set-effect interactions by translation side", "lrrrl")
print("Appendix exports written.")

Appendix exports written.


## 9. Paper-ready numerical summary

The summary and claim-support registry below are generated programmatically from the validated main tables and contrasts.

In [6]:
best_row = table1.loc[table1["Set AUPRC"].idxmax()]
target_nllb = contrast_row("en_set_ph_en_nllb_vs_pt", "auprc", family="target_translation")
target_madlad = contrast_row("en_set_ph_en_madlad_vs_pt", "auprc", family="target_translation")
training_nllb = contrast_row("pt_nllb_set_vs_en_ph_pt", "auprc", family="training_translation")
training_madlad = contrast_row("pt_madlad_set_vs_en_ph_pt", "auprc", family="training_translation")
interaction_nllb = contrast_row("nllb_set_target_minus_training", "auprc", regime="interaction", family="translation_interaction")
interaction_madlad = contrast_row("madlad_set_target_minus_training", "auprc", regime="interaction", family="translation_interaction")
higher_count = int((table2["Delta Set-Attention"] > 0).sum())
favorable_count = int((table2["_ci_low"] > 0).sum())

paper_summary = "\n".join([
    "# Paper-ready numerical summary", "",
    "## Best direct transfer",
    f"- EN + Set → PH PT: AUPRC {fmt(best_row['Set AUPRC'])}; AUROC {fmt(best_row['Set AUROC'])}.", "",
    "## Set effect",
    f"- Set has higher AUPRC in {higher_count}/5 main conditions.",
    f"- The paired 95% CI favors Set in {favorable_count}/5 conditions; {5 - favorable_count} is inconclusive.", "",
    "## Target translation effect — Set",
    f"- NLLB: delta {fmt_signed(target_nllb['observed_delta'])}, 95% CI {ci_text(target_nllb)}.",
    f"- MADLAD: delta {fmt_signed(target_madlad['observed_delta'])}, 95% CI {ci_text(target_madlad)}.", "",
    "## Training translation effect — Set",
    f"- NLLB: delta {fmt_signed(training_nllb['observed_delta'])}, 95% CI {ci_text(training_nllb)}.",
    f"- MADLAD: delta {fmt_signed(training_madlad['observed_delta'])}, 95% CI {ci_text(training_madlad)}.", "",
    "## Translation-side interaction",
    f"- NLLB AUPRC interaction: {fmt_signed(interaction_nllb['observed_delta'])}, 95% CI {ci_text(interaction_nllb)}; target-side more harmful.",
    f"- MADLAD AUPRC interaction: {fmt_signed(interaction_madlad['observed_delta'])}, 95% CI {ci_text(interaction_madlad)}; target-side more harmful.", "",
    "These are associations under the frozen paired protocol; no causal interpretation is implied.",
]) + "\n"
(OUTPUT_ROOT / "paper_ready_summary.md").write_text(paper_summary)
display(Markdown(paper_summary))

claim_support = pd.DataFrame([
    {"claim_id": "C1", "claim": "Direct EN->PT transfer works well", "evidence": f"EN + Set is the highest-AUPRC main condition ({fmt(best_row['Set AUPRC'])}).", "statistical_status": "SUPPORTED", "allowed_wording": "Direct EN→PT transfer is the strongest evaluated transfer strategy.", "destination": "Results"},
    {"claim_id": "C2", "claim": "Set has higher AUPRC across the main conditions", "evidence": f"Higher point estimate in {higher_count}/5; paired CI favors Set in {favorable_count}/5.", "statistical_status": "SUPPORTED", "allowed_wording": "Set has higher AUPRC in all five main conditions, with paired CIs favoring Set in four.", "destination": "Results"},
    {"claim_id": "C3", "claim": "Training translation does not provide consistent overall improvement over EN", "evidence": f"NLLB {fmt_signed(training_nllb['observed_delta'])} {ci_text(training_nllb)}; MADLAD {fmt_signed(training_madlad['observed_delta'])} {ci_text(training_madlad)}.", "statistical_status": "SUPPORTED", "allowed_wording": "Explicit training translation does not provide a consistent improvement over EN training.", "destination": "Discussion"},
    {"claim_id": "C4", "claim": "Target translation substantially degrades transfer", "evidence": f"Set target deltas: NLLB {fmt_signed(target_nllb['observed_delta'])} {ci_text(target_nllb)}; MADLAD {fmt_signed(target_madlad['observed_delta'])} {ci_text(target_madlad)}.", "statistical_status": "SUPPORTED", "allowed_wording": "Target translation produces large ranking degradation for both translators.", "destination": "Results"},
    {"claim_id": "C5", "claim": "Target-side translation is more harmful than training-side translation", "evidence": f"Set interactions: NLLB {fmt_signed(interaction_nllb['observed_delta'])} {ci_text(interaction_nllb)}; MADLAD {fmt_signed(interaction_madlad['observed_delta'])} {ci_text(interaction_madlad)}.", "statistical_status": "SUPPORTED", "allowed_wording": "Target-side translation is associated with larger degradation than training-side translation.", "destination": "Results"},
    {"claim_id": "C6", "claim": "EN+Set approaches target-supervised Attention descriptively", "evidence": f"EN + Set ({fmt(table4.loc[table4['Method'].eq('EN + Set'), 'AUPRC'].iloc[0])}/{fmt(table4.loc[table4['Method'].eq('EN + Set'), 'AUROC'].iloc[0])}) and target-supervised Attention ({fmt(table4.loc[table4['Method'].eq('Target-supervised Attention'), 'AUPRC'].iloc[0])}/{fmt(table4.loc[table4['Method'].eq('Target-supervised Attention'), 'AUROC'].iloc[0])}).", "statistical_status": "DESCRIPTIVE_ONLY", "allowed_wording": "EN+Set approaches the target-supervised Attention result descriptively; no paired equivalence or superiority claim is made.", "destination": "Results"},
])
assert claim_support.loc[claim_support["claim_id"].eq("C6"), "statistical_status"].iloc[0] == "DESCRIPTIVE_ONLY"
claim_support.to_csv(OUTPUT_ROOT / "claim_support.csv", index=False)
display(claim_support[["claim_id", "statistical_status", "allowed_wording"]])

# Paper-ready numerical summary

## Best direct transfer
- EN + Set → PH PT: AUPRC 0.6035; AUROC 0.8862.

## Set effect
- Set has higher AUPRC in 5/5 main conditions.
- The paired 95% CI favors Set in 4/5 conditions; 1 is inconclusive.

## Target translation effect — Set
- NLLB: delta -0.1973, 95% CI [-0.2270, -0.1633].
- MADLAD: delta -0.3703, 95% CI [-0.4098, -0.3250].

## Training translation effect — Set
- NLLB: delta -0.0377, 95% CI [-0.0587, -0.0178].
- MADLAD: delta -0.0148, 95% CI [-0.0336, 0.0020].

## Translation-side interaction
- NLLB AUPRC interaction: -0.1596, 95% CI [-0.1914, -0.1224]; target-side more harmful.
- MADLAD AUPRC interaction: -0.3555, 95% CI [-0.3952, -0.3082]; target-side more harmful.

These are associations under the frozen paired protocol; no causal interpretation is implied.


,claim_id,statistical_status,allowed_wording
0,C1,SUPPORTED,Direct EN→PT transfer is the strongest evaluat...
1,C2,SUPPORTED,Set has higher AUPRC in all five main conditio...
2,C3,SUPPORTED,Explicit training translation does not provide...
3,C4,SUPPORTED,Target translation produces large ranking degr...
4,C5,SUPPORTED,Target-side translation is associated with lar...
5,C6,DESCRIPTIVE_ONLY,EN+Set approaches the target-supervised Attent...


## 10. Export inventory and reproducibility checks

The final cell validates the frozen scientific relationships, checks every requested output, and records the deterministic inventory.

In [7]:
assert int((table1["Set AUPRC"] >= table1["Attention AUPRC"]).sum()) == 5
assert int((table2["_ci_low"] > 0).sum()) == 4
assert int(((table2["_ci_low"] <= 0) & (table2["_ci_high"] >= 0)).sum()) == 1
assert target_nllb["observed_delta"] < 0 and target_nllb["ci_high"] < 0
assert target_madlad["observed_delta"] < 0 and target_madlad["ci_high"] < 0
assert training_nllb["observed_delta"] < 0 and training_nllb["ci_high"] < 0
assert training_madlad["observed_delta"] < 0 and training_madlad["ci_low"] < 0 < training_madlad["ci_high"]
assert interaction_nllb["observed_delta"] < 0 and interaction_nllb["ci_high"] < 0
assert interaction_madlad["observed_delta"] < 0 and interaction_madlad["ci_high"] < 0
assert claim_support.loc[claim_support["claim_id"].eq("C6"), "statistical_status"].iloc[0] == "DESCRIPTIVE_ONLY"

expected_outputs = [
    TABLE_ROOT / "table_main_transfer.csv", TABLE_ROOT / "table_main_transfer.tex",
    FIGURE_ROOT / "figure_transfer_overview.pdf", FIGURE_ROOT / "figure_transfer_overview.png",
    TABLE_ROOT / "table_set_vs_attention.csv", TABLE_ROOT / "table_set_vs_attention.tex",
    FIGURE_ROOT / "figure_translation_effects.pdf", FIGURE_ROOT / "figure_translation_effects.png",
    TABLE_ROOT / "table_translation_interaction.csv", TABLE_ROOT / "table_translation_interaction.tex",
    TABLE_ROOT / "table_translation_interaction_full.csv", TABLE_ROOT / "table_translation_interaction_full.tex",
    TABLE_ROOT / "table_baselines.csv", TABLE_ROOT / "table_baselines.tex",
    APPENDIX_ROOT / "table_pooling_ablations.csv", APPENDIX_ROOT / "table_pooling_ablations.tex",
    APPENDIX_ROOT / "table_negative_ablations.csv", APPENDIX_ROOT / "table_negative_ablations.tex",
    APPENDIX_ROOT / "table_brier_main_conditions.csv", APPENDIX_ROOT / "table_brier_main_conditions.tex",
    APPENDIX_ROOT / "table_thresholded_results.csv", APPENDIX_ROOT / "table_thresholded_results.tex",
    APPENDIX_ROOT / "table_full_paired_bootstrap.csv", APPENDIX_ROOT / "table_full_paired_bootstrap.tex",
    APPENDIX_ROOT / "table_set_effect_interactions.csv", APPENDIX_ROOT / "table_set_effect_interactions.tex",
    OUTPUT_ROOT / "paper_ready_summary.md", OUTPUT_ROOT / "claim_support.csv",
]
assert all(path.exists() and path.stat().st_size > 0 for path in expected_outputs)
inventory = pd.DataFrame([
    {"path": str(path.relative_to(ROOT)), "type": path.suffix.lstrip("."),
     "intended destination": "Main" if path.parent in {TABLE_ROOT, FIGURE_ROOT} else "Appendix" if path.parent == APPENDIX_ROOT else "Summary",
     "status": "generated"}
    for path in expected_outputs
])
display(inventory)
print("PASS: all scientific consistency assertions passed.")
print("restart-and-run-all: deterministic; outputs overwrite the same paths.")
print("scientific artifacts modified: NO")
print("bootstrap executed: NO")
print("thresholds recomputed: NO")
print("predictions evaluated: NO")

,path,type,intended destination,status
0,results/paper/tables/table_main_transfer.csv,csv,Main,generated
1,results/paper/tables/table_main_transfer.tex,tex,Main,generated
2,results/paper/figures/figure_transfer_overview...,pdf,Main,generated
3,results/paper/figures/figure_transfer_overview...,png,Main,generated
4,results/paper/tables/table_set_vs_attention.csv,csv,Main,generated
5,results/paper/tables/table_set_vs_attention.tex,tex,Main,generated
6,results/paper/figures/figure_translation_effec...,pdf,Main,generated
7,results/paper/figures/figure_translation_effec...,png,Main,generated
8,results/paper/tables/table_translation_interac...,csv,Main,generated
9,results/paper/tables/table_translation_interac...,tex,Main,generated


PASS: all scientific consistency assertions passed.
restart-and-run-all: deterministic; outputs overwrite the same paths.
scientific artifacts modified: NO
bootstrap executed: NO
thresholds recomputed: NO
predictions evaluated: NO
